# 82 · Published human vs mouse PT findings in independent data: Cheval 2012 and single-nucleus atlases

Published comparisons of human and mouse kidney report level differences inside one segment, or whole-tissue
differences. Cheval et al. 2012 microdissected S1 and S3 in both species (SAGE) and concluded that expression
diverges, but never compared the within-species S1 → S3 contrast. This notebook re-reads that question, and notebook
79's zonation transfer calls, in data that are not ours.

| Part | Question | Data |
|---|---|---|
| B1 | Is the S1 → S3 contrast conserved between species, and does conservation rise with gradient strength? | Cheval only |
| B2 | Do Cheval's contrasts agree with ours, and does our species × position interaction replicate? | Cheval and our pseudobulks |
| B2c | Do 79's zonation transfer calls hold in Cheval? | Cheval |
| B3 | Do 79's zonation transfer calls hold in single-nucleus atlases, with donors as replicates? | Census mouse, Lake human, Census human |
| A3 (atlas half) | Do CellSpectra's conserved PT functions peak in the same segment in the atlases as in our data? | Census mouse, Lake human |

**Protocol.** `docs/results/pt-published-findings.md` (commit 0a44dec), frozen before any of these analyses ran.

**What bounds the reading.**
- Cheval has one pooled library per structure and species, so B1 and B2c have genes, not donors, as replicates.
- SAGE depth is low: only genes with at least 10 tags over S1 and S3 in each species are eligible.
- Our data enter only through notebook 79's pseudobulks (recomputed here and checked against 79's table).
- Cohort: two male control mice and two cortex sections of one male human donor. Everything is descriptive except
  B2b, which is one pre-specified test.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
NATIVE_COR1 = results_root / 'package_pseudospace' / 'runs' / 'hvm' / '01_units' / 'HUK1_COR1_units.h5ad'   # Lake id map
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only
T75 = results_root / 'pt_package_pathway_methods'                 # read only: the Lake S1/S2/S3 cortex cache
T79 = results_root / 'pt_known_species_programs'                  # read only: pseudobulks and classes
nb38 = results_root / 'pt_literature_deep'
LAKE_NPZ = nb38 / 'zonation_check' / 'lake_cortex_s1_s3_all_disease.npz'
LAKE_DONORS = nb38 / 'zonation_check' / 'lake_donor_amplitude.csv'
KNOWN = project / 'docs' / 'results' / 'known-programs'           # 79's frozen programs
LIT = project / 'docs' / 'results' / 'published-findings'         # this protocol's frozen programs
CHEVAL = data_root / 'external' / 'cheval2012'
CENSUS = data_root / 'external' / 'census_pt_segments'
output = results_root / 'pt_published_findings_external'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

SHA = {KNOWN / 'programs_hvm.csv': 'be61b8183c51d798ee33314e6226d5d8dd269188b82eb60ae65c888a355db79e',
       KNOWN / 'programs_hvm_members.csv': '2a3889075ca4130b102661409aee1390f1dd1f936a4171c9c9b7b0a7365299bf',
       LIT / 'programs_lit.csv': 'feeee7503b3b522df5876b01c17855ff0e1bcd39430867cec7f2f623043a6e4e',
       LIT / 'programs_lit_members.csv': '8bcbd2556f52b859ad4ef62ffd2c98c14d4821e784a68ec979ecfbebbab219c7',
       CHEVAL / 'cheval2012_table_s1.csv': '52ea96118eb2e3744c22515aa9c6a8bbee150a188d520d9e49c870100d6f7fb9',
       CHEVAL / 'cheval2012_mouse_symbols.csv': '33f65262448c0d53c52f5453bf84a3637b4834be41150a643fd2ad4142e4d30d',
       CENSUS / 'mouse_pt_segment_pseudobulk_counts.csv.gz': 'd8bb110cd86c3d47c2d627de71edc17f5ad1634c6b9515e04162666adb0871bf',
       CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz': 'fcca3c628b94d19057d9c821748260900788b1f23d0365838b9efe95d75d5f47'}
REQUIRED = [EXPORT, NATIVE_COR1, upstream13 / 'ortholog_map_used.csv', LAKE_NPZ, LAKE_DONORS, *SHA,
            T71 / 'tables' / 'inputs.csv', T71 / 'tables' / 'universe_human_vs_mouse.csv',
            T79 / 'tables' / 'pseudobulk_rates.csv.gz', T79 / 'tables' / 'classes.csv',
            CENSUS / 'mouse_pt_segment_pseudobulk_meta.csv', CENSUS / 'human_pt_segment_pseudobulk_meta.csv',
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 71, 75 and 79 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '82.published_findings_external.1'   # Bump when a calculation changes.
FINGERPRINT = 'efafc9ac2e99'                                    # confirmed hvm pass-1 labels (notebook 70)
FLOOR = .5                                                      # 79's pseudobulk floor
SEGMENTS = ('S1', 'S2', 'S3')
SCOPES = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3'], 'PT': None}  # 79's scopes (the G3 guard compares all four)
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE, HUMAN = NAMES[:2], NAMES[2:]
SPECIMENS = {'mouse': MOUSE, 'human': HUMAN}
LABEL_ORIENTATION = {'Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b'}     # 79's R3 (72's exclusion)
CHEVAL_USED = ['human_S1', 'human_S3', 'mouse_S1', 'mouse_S3']                 # only S1 and S3 enter
CHEVAL_STRUCTURES = ['Glom', 'S1', 'S3', 'mTAL', 'cTAL', 'DCT', 'CCD', 'OMCD']
ELIGIBLE_MIN, ZON_EFFECT, ZON_Z = 10, .5, 2.                                    # protocol B (tags; log2; |d|/SE)
STRENGTH_BINS = (.5, 1., 2., np.inf)                                            # B1a
N_BOOT, SEED_B1 = 2000, 81                                                      # B1b
N_PERM, SEED_B2 = 10000, 82                                                     # B2b
MIN_MEMBERS, MIN_CELLS, ALPHA = 5, 50, .05
B2C_PROGRAMS = ['Z-M1', 'Z-M3', 'Z-H1', 'Z-H3']                                 # no S2 in Cheval
Z_PROGRAMS = ['Z-M1', 'Z-M2', 'Z-M3', 'Z-H1', 'Z-H2', 'Z-H3']
FUNCTIONS = ['F-LCFA', 'F-ARG', 'F-PLB', 'F-OAT', 'F-TCY', 'F-LDL']             # A3
MOUSE_STAGES = ('3-6 month-old mature stage', '12-month-old stage')            # notebook 74's six-donor set
CENSUS_MOUSE_SEGMENTS = {'epithelial cell of proximal tubule segment 1': 'S1',
                         'epithelial cell of proximal tubule segment 2': 'S2',
                         'epithelial cell of proximal tubule segment 3': 'S3'}
CENSUS_HUMAN_SEGMENTS = {'kidney proximal convoluted tubule epithelial cell': 'PCT',
                         'epithelial cell of proximal tubule segment 3': 'S3'}
LAKE_RECORD = ['164-10', '164-6', '18-142', '18-312', '3535', 'KRP446', 'KRP460']   # 38's primary donors (pt-gene-model-nb.md)

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
MOUSE_COLOR, HUMAN_COLOR = '#0072B2', '#D55E00'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 120)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def relative(path):
    """A path relative to the repository, the data root or the results root (never absolute in outputs)."""
    for root, name in ((data_root, 'DATA'), (results_root, 'RESULTS'), (project, 'REPO')):
        try:
            return f'{name}/{Path(path).relative_to(root)}'
        except ValueError:
            continue
    return Path(path).name


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Our inputs and guards

Our data enter only as notebook 79's specimen × segment pseudobulks. They are rebuilt exactly as 79 builds them
(`package_inputs.hvm_inputs`, the 13 `mt-` genes out of library and universe). Guards (not cached):
- **G1:** the export contract and the confirmed pass-1 fingerprint; the label/orientation genes equal the protocol's 15.
- **G2:** 71's universe (11,055 genes) and units (11,383).
- **G3:** the recomputed pseudobulks equal 79's `pseudobulk_rates.csv.gz`; the program medians of our segment
  contrasts reproduce 79's zonation contrasts.
- **G4:** the program files (79's and this protocol's) match their frozen sha256; every member is in the universe
  and none is a label/orientation gene.
- **G5:** the external files match the protocol's sha256.

In [ ]:
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace import program_localization as pl
from pseudospace import published_findings as pf

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')                                                       # G1
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
panels = PROV['config']['marker_panels_yaml']
label_orientation = ({str(g) for s in ('PT-S1', 'PT-S2', 'PT-S3') for g in panels['label_panels'][s]}
                     | {str(g) for side in ('early', 'late') for g in panels['axis_anchors'][side]})
assert label_orientation == LABEL_ORIENTATION, label_orientation ^ LABEL_ORIENTATION
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
GENE_SET = set(GENES)
universe71 = pd.read_csv(T71 / 'tables' / 'universe_human_vs_mouse.csv')
universe71 = universe71[universe71.eligible.astype(bool)]
assert GENES == universe71.gene.astype(str).tolist(), 'G2: the universe differs from notebook 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = pd.Series(INP['segment']).str.replace('PT-', '', regex=False).to_numpy()
COUNTS = INP['counts'].tocsr()
LIBRARY = np.asarray(INP['library'], float)
GUARDS.append(f'G1/G2: export contract; pass-1 fingerprint {FINGERPRINT}; universe ({len(GENES):,}) and units '
              f"({len(SPECIMEN):,}) equal notebook 71's; the label/orientation genes equal the protocol's 15")

RATES = pl.segment_pseudobulk(COUNTS, LIBRARY, SPECIMEN, SEGMENT, SCOPES, genes=GENES, floor=FLOOR)
saved79 = pd.read_csv(T79 / 'tables' / 'pseudobulk_rates.csv.gz', header=[0, 1], index_col=0)
again = pd.concat({s: RATES[s] for s in SCOPES}, axis=1)
assert saved79.index.astype(str).tolist() == GENES and list(saved79.columns) == list(again.columns)
np.testing.assert_allclose(again.to_numpy(), saved79.to_numpy(), rtol=0, atol=1e-12)                     # G3
GUARDS.append(f"G3: the recomputed specimen × scope pseudobulks equal notebook 79's pseudobulk_rates.csv.gz "
              f'({again.shape[0]:,} genes × {again.shape[1]} columns; max |difference| '
              f'{np.abs(again.to_numpy() - saved79.to_numpy()).max():.1e})')

# Our S3 − S1 per specimen (79 section 3.2: centred by its median over the universe genes); species = mean of two.
S3_S1 = pd.DataFrame({n: RATES['S3'][n] - RATES['S1'][n] for n in NAMES})
S3_S1 = S3_S1 - S3_S1.median()
OURS = pd.DataFrame({sp: S3_S1[SPECIMENS[sp]].mean(axis=1) for sp in SPECIMENS})
# Our segment contrasts (79 section 3.4): per specimen, segment minus the mean of the others, median-centred.
ZON = {(sp, seg): pl.segment_contrast({s: RATES[s] for s in SEGMENTS}, SPECIMENS[sp], seg)
       for sp in SPECIMENS for seg in SEGMENTS}

for path, value in SHA.items():                                                                          # G4/G5
    assert sha256(path) == value, f'{relative(path)}: sha256 differs from the protocol'
Z79, META79 = pl.read_programs(KNOWN / 'programs_hvm.csv', GENES, LABEL_ORIENTATION, sha256=SHA[KNOWN / 'programs_hvm.csv'])
FN, META_LIT = pl.read_programs(LIT / 'programs_lit.csv', GENES, LABEL_ORIENTATION, sha256=SHA[LIT / 'programs_lit.csv'])
for programs, path in ((Z79, KNOWN / 'programs_hvm_members.csv'), (FN, LIT / 'programs_lit_members.csv')):
    members = pd.read_csv(path)
    assert all(members[members.program_id.eq(p)].gene.tolist() == v for p, v in programs.items()), path.name
assert all(p in Z79 for p in Z_PROGRAMS) and all(f in FN for f in FUNCTIONS)
assert [len(FN[f]) for f in FUNCTIONS] == [10, 8, 19, 209, 18, 13], [len(FN[f]) for f in FUNCTIONS]
GUARDS.append('G4: the program files (79: programs_hvm; this protocol: programs_lit) match their frozen sha256; every '
              'member is in the universe and none is a label/orientation gene; the A3 functions have 10, 8, 19, 209, 18 '
              'and 13 members')
GUARDS.append('G5: the Cheval table, its symbol pairing and the Census mouse and human count files match the '
              "protocol's sha256")

CLASSES79 = pd.read_csv(T79 / 'tables' / 'classes.csv', index_col=0)
SEG = {p: str(META79.loc[p, 'segment']) for p in Z_PROGRAMS}
SOURCE = {p: str(CLASSES79.loc[p, 'source species']) for p in Z_PROGRAMS}
OTHER = {p: 'human' if SOURCE[p] == 'mouse' else 'mouse' for p in Z_PROGRAMS}
SOURCE79 = {p: str(CLASSES79.loc[p, 'replication']) for p in Z_PROGRAMS}       # 79's positive control (zonated)
TRANSFER79 = {p: str(CLASSES79.loc[p, 'transfer']) for p in Z_PROGRAMS}


def our_program(program_members, species, seg):
    """Per specimen of a species: the program median of its centred segment contrast (79 section 3.4)."""
    frame = ZON[(species, seg)]
    return [float(np.median(frame.loc[program_members, s])) for s in SPECIMENS[species]]


for p in Z_PROGRAMS:                                                                                      # G3b
    for role, species in (('source', SOURCE[p]), ('other', OTHER[p])):
        np.testing.assert_allclose(our_program(Z79[p], species, SEG[p]), json.loads(CLASSES79.loc[p, f'{role} contrasts']),
                                   rtol=1e-9, atol=1e-12)
GUARDS.append("G3b: the program medians of our segment contrasts reproduce notebook 79's source and transfer contrasts "
              'for the six zonation programs')
display(pd.DataFrame({'segment': SEG, 'source species': SOURCE, "79 source class": SOURCE79, "79 transfer class": TRANSFER79,
                      'members': {p: len(Z79[p]) for p in Z_PROGRAMS}}))
print('\n'.join('· ' + g for g in GUARDS))
print(f'Our inputs {time.perf_counter() - t0:.0f} s')

## 3 · Cheval 2012: SAGE counts and S3 − S1 contrasts (protocol B)

- **Counts.** One tag is the column's smallest non-zero value, so c = value / unit and the library is
  N = 10,000 / unit (`published_findings.sage_counts`). Only the S1 and S3 columns of each species are used.
- **Pairing.** The authors' human → mouse symbol table. Rows without a mouse symbol are left out.
- **Several rows for one gene.** Within a gene and species, rows that repeat the same 8-structure values are the
  same tag listed twice and count once; distinct rows are summed.
- **Contrast** d = log2((c_S3 + 0.5)/N_S3) − log2((c_S1 + 0.5)/N_S1), with Poisson SE (`sage_contrast`).
- **Eligible:** c_S1 + c_S3 ≥ 10 in each species. **Zonated** in a species: |d| ≥ 0.5 and |d|/SE ≥ 2.

A sensitivity (section 9) takes one tag as each column's most frequent non-zero value instead of its smallest.

In [ ]:
from scipy.stats import binomtest

raw = pd.read_csv(CHEVAL / 'cheval2012_table_s1.csv')
symbols = pd.read_csv(CHEVAL / 'cheval2012_mouse_symbols.csv')
assert len(raw) == len(symbols) and (raw.human_symbol.to_numpy() == symbols.human_symbol.to_numpy()).all()
assert raw[CHEVAL_USED].notna().all().all() and (raw[CHEVAL_USED] >= 0).all().all()
raw['mouse_symbol'] = symbols.mouse_symbol
paired = raw[raw.mouse_symbol.notna()].copy()
assert (paired.groupby('mouse_symbol').human_symbol.nunique() == 1).all()                      # one-to-one pairing
assert (paired.groupby('human_symbol').mouse_symbol.nunique() == 1).all()
VALUE_COLUMNS = {sp: [f'{sp}_{s}' for s in CHEVAL_STRUCTURES] for sp in ('human', 'mouse')}
DISTINCT_ROWS = {sp: paired.index[~paired.duplicated(subset=['mouse_symbol'] + VALUE_COLUMNS[sp])] for sp in VALUE_COLUMNS}
HUMAN_OF = paired.drop_duplicates('mouse_symbol').set_index('mouse_symbol').human_symbol


def cheval_genes(row_counts, library):
    """Per paired gene: counts, contrasts, SE, zonation per species and eligibility (protocol B)."""
    cols = {}
    for sp, rows in DISTINCT_ROWS.items():
        for seg in ('S1', 'S3'):
            cols[f'c_{sp}_{seg}'] = row_counts.loc[rows, f'{sp}_{seg}'].groupby(paired.loc[rows, 'mouse_symbol']).sum()
    g = pd.DataFrame(cols)
    g.index.name = 'mouse_symbol'
    g.insert(0, 'human_symbol', HUMAN_OF.reindex(g.index))
    for sp in ('human', 'mouse'):
        d, se = pf.sage_contrast(g[f'c_{sp}_S3'], g[f'c_{sp}_S1'], library[f'{sp}_S3'], library[f'{sp}_S1'])
        g[f'd_{sp}'], g[f'se_{sp}'] = d, se
        g[f'zonated_{sp}'] = pf.zonated(d, se, min_effect=ZON_EFFECT, min_z=ZON_Z)
    g['eligible'] = ((g.c_human_S1 + g.c_human_S3) >= ELIGIBLE_MIN) & ((g.c_mouse_S1 + g.c_mouse_S3) >= ELIGIBLE_MIN)
    return g


def sage_counts_modal(values):
    """SENSITIVITY ONLY (notebook-local; not the protocol's rule): one tag = the column's most frequent non-zero value.

    The protocol's rule (``published_findings.sage_counts``) takes the smallest non-zero value. In every used column a
    few values lie below the modal one-tag value, so the smallest value is not one tag; this arm asks whether that
    matters. Ceiling: the mode is rounded to 2 decimals as published.
    """
    values = pd.DataFrame(values, dtype=float)
    unit = values.apply(lambda c: c[c > 0].round(4).mode().min())
    return (values / unit).round(), 1e4 / unit


ROW_COUNTS, CHEVAL_LIBRARY = pf.sage_counts(raw[CHEVAL_USED])
CHEVAL = cheval_genes(ROW_COUNTS, CHEVAL_LIBRARY)
ROW_COUNTS_MODAL, CHEVAL_LIBRARY_MODAL = sage_counts_modal(raw[CHEVAL_USED])
CHEVAL_MODAL = cheval_genes(ROW_COUNTS_MODAL, CHEVAL_LIBRARY_MODAL)
unit_table = pd.DataFrame({'unit: smallest non-zero (protocol)': 1e4 / CHEVAL_LIBRARY,
                           'unit: most frequent non-zero (sensitivity)': 1e4 / CHEVAL_LIBRARY_MODAL,
                           'non-zero values below the modal unit': [int(((raw[c] > 0) & (raw[c] < 1e4 / CHEVAL_LIBRARY_MODAL[c] - 1e-9)).sum()) for c in CHEVAL_USED],
                           'library N (protocol)': CHEVAL_LIBRARY, 'tags in the table (protocol)': ROW_COUNTS.sum()})
display(unit_table)
cheval_counts = {'table rows': len(raw), 'rows without a mouse symbol': int(raw.mouse_symbol.isna().sum()),
                 'paired genes': len(CHEVAL), 'genes with several rows': int(paired.mouse_symbol.duplicated().groupby(paired.mouse_symbol).any().sum()),
                 'eligible': int(CHEVAL.eligible.sum()),
                 'eligible, zonated in mouse': int((CHEVAL.eligible & CHEVAL.zonated_mouse).sum()),
                 'eligible, zonated in human': int((CHEVAL.eligible & CHEVAL.zonated_human).sum()),
                 'eligible ∩ universe': int(CHEVAL.eligible[CHEVAL.index.isin(GENE_SET)].sum())}
print(cheval_counts)
READING.append('Cheval: only the S1 and S3 columns of each species enter. One tag is the smallest non-zero value of each '
               'column over all 4,644 published rows (the protocol rule); rows without a mouse symbol in the authors\' '
               'pairing table are left out of every analysis, B1 included.')
READING.append('Cheval genes listed in several rows: within a gene and species, rows that repeat the same 8-structure '
               'values are one tag listed twice and count once; distinct rows are summed (tags of one gene). The pairing '
               'table is one-to-one at the symbol level.')

## 4 · B1 · Their data, our question: is the S1 → S3 contrast conserved, and does conservation rise with strength?

- **B1a.** Among eligible genes zonated in mouse, the share with the same sign in human, by mouse |d| bins
  [0.5, 1), [1, 2), [2, ∞), and the mirror (human-zonated genes by human |d|). Binomial p against 0.5 per bin and a
  one-sided Cochran–Armitage trend p. **Hypothesis:** agreement > 0.5 and rising with strength, in both directions.
- **B1b.** Slopes through the origin: human d on mouse d (mouse-zonated genes) and mouse d on human d
  (human-zonated genes), with a 2,000-draw gene bootstrap (seed 81). **Hypothesis:** human-on-mouse < mouse-on-human
  (human flatter). Descriptive, beside the atlas values 0.12 and 0.62.

In [ ]:
def b1(g):
    """B1a per-bin agreement with trend, and B1b slopes, for one Cheval gene table."""
    e = g[g.eligible]
    bins, slopes, verdicts = [], [], {}
    for sel, other in (('mouse', 'human'), ('human', 'mouse')):
        z = e[f'zonated_{sel}'].to_numpy()
        frame, (tz, tp) = pf.strength_agreement(e[f'd_{sel}'].to_numpy(), e[f'd_{other}'].to_numpy(), z, bins=STRENGTH_BINS)
        n, k = int(frame.n.sum()), int(frame.agree.sum())
        pooled_p = float(binomtest(k, n, .5, alternative='greater').pvalue) if n else np.nan
        frame.insert(0, 'selection', f'zonated in {sel}')
        frame['ci low'] = [binomtest(int(a), int(m)).proportion_ci().low if m else np.nan for a, m in zip(frame.agree, frame.n)]
        frame['ci high'] = [binomtest(int(a), int(m)).proportion_ci().high if m else np.nan for a, m in zip(frame.agree, frame.n)]
        frame = frame.assign(**{'pooled n': n, 'pooled share': k / n if n else np.nan, 'pooled p (> 0.5)': pooled_p,
                                'trend z': tz, 'trend p (rising)': tp})
        bins.append(frame)
        above = bool((frame.share[frame.n > 0] > .5).all()) and pooled_p <= ALPHA
        rising = bool(np.isfinite(tp) and tp <= ALPHA)
        verdicts[sel] = {'agreement > 0.5': above, 'rising with strength': rising}
        est, lo, hi = pf.slope_through_origin(e.loc[z, f'd_{sel}'].to_numpy(), e.loc[z, f'd_{other}'].to_numpy(),
                                              n_boot=N_BOOT, seed=SEED_B1)
        slopes.append({'slope': f'{other} d on {sel} d', 'selection': f'zonated in {sel}', 'genes': int(z.sum()),
                       'estimate': est, 'ci low': lo, 'ci high': hi})
    slopes = pd.DataFrame(slopes).set_index('slope')
    hm, mh = slopes.loc['human d on mouse d'], slopes.loc['mouse d on human d']
    verdicts['B1b'] = {'human flatter (point estimates)': bool(hm.estimate < mh.estimate),
                       'intervals separate': bool(hm['ci high'] < mh['ci low'])}
    return pd.concat(bins, ignore_index=True), slopes, verdicts


B1_BINS, B1_SLOPES, B1_VERDICT = b1(CHEVAL)
display(B1_BINS.round(4))
display(B1_SLOPES.round(3))
for key, value in B1_VERDICT.items():
    print(f'B1 {key}: {value}')
READING.append('B1a "agreement > 0.5" is read as: every non-empty bin above 0.5 and the pooled one-sided binomial p over '
               'all selected genes ≤ 0.05; "rising" as the one-sided Cochran–Armitage trend p ≤ 0.05 (bin scores 0, 1, 2). '
               'A gene whose other-species d is exactly 0 counts as disagreeing (the module rule).')
READING.append('B1b is read on point estimates ("human flatter" when the human-on-mouse slope is below the mouse-on-human '
               'slope); the two bootstrap intervals are reported, and "intervals separate" is descriptive.')

## 5 · B2 · Cheval against our data

Genes: Cheval-eligible ∩ 71's universe, minus the 15 label/orientation genes. Ours: per species, the mean over its
two specimens of the median-centred pseudobulk r_S3 − r_S1.
- **B2a (positive controls):** Spearman of Cheval's mouse d with our mouse S3 − S1, and of Cheval's human d with
  our human S3 − S1.
- **B2b (confirmatory, one test):** x = Cheval (d_human − d_mouse), y = ours (human − mouse). Spearman ρ with a
  one-sided permutation p (10,000; seed 82). **Beyond flattening:** the rank-partial ρ given both mouse contrasts
  (Cheval's and ours), with the same permutation. "Replicates" when ρ > 0 and p ≤ 0.05; "replicates beyond
  flattening" when the partial also has p ≤ 0.05.

In [ ]:
def versus_ours(g):
    genes = [x for x in g.index[g.eligible] if x in GENE_SET and x not in LABEL_ORIENTATION]
    c, o = g.loc[genes], OURS.loc[genes]
    rows = []
    for sp in ('mouse', 'human'):
        rho, p, n = pf.spearman(c[f'd_{sp}'], o[sp])
        rows.append({'readout': f'B2a {sp}: Cheval d vs our S3 − S1', 'genes': n, 'rho': rho, 'p': p, 'p kind': 'Spearman (two-sided)'})
    x = (c.d_human - c.d_mouse).to_numpy()
    y = (o.human - o.mouse).to_numpy()
    rho, p = pf.permutation_spearman(x, y, n_perm=N_PERM, seed=SEED_B2)
    cov = np.column_stack([c.d_mouse.to_numpy(), o.mouse.to_numpy()])
    rho_b, p_b = pf.permutation_spearman(x, y, covariates=cov, n_perm=N_PERM, seed=SEED_B2)
    raw_claim = rho > 0 and p <= ALPHA
    rows.append({'readout': 'B2b interaction (raw)', 'genes': len(genes), 'rho': rho, 'p': p,
                 'p kind': f'one-sided permutation ({N_PERM:,}; seed {SEED_B2})', 'claim': 'replicates' if raw_claim else 'does not replicate'})
    rows.append({'readout': 'B2b interaction beyond flattening (partial | Cheval and our mouse contrasts)', 'genes': len(genes),
                 'rho': rho_b, 'p': p_b, 'p kind': f'one-sided residual permutation ({N_PERM:,}; seed {SEED_B2})',
                 'claim': 'replicates beyond flattening' if raw_claim and p_b <= ALPHA else 'not beyond flattening'})
    frame = pd.DataFrame({'cheval d_mouse': c.d_mouse, 'cheval d_human': c.d_human, 'our mouse S3−S1': o.mouse,
                          'our human S3−S1': o.human, 'cheval interaction': x, 'our interaction': y}, index=genes)
    return pd.DataFrame(rows), frame


B2, B2_GENES = versus_ours(CHEVAL)
display(B2.round(4))
# Descriptive context: the flattening slope (human on mouse, through the origin) in each data set on the B2 genes.
B2_SLOPES = {'Cheval': float((B2_GENES['cheval d_mouse'] * B2_GENES['cheval d_human']).sum() / (B2_GENES['cheval d_mouse'] ** 2).sum()),
             'ours': float((B2_GENES['our mouse S3−S1'] * B2_GENES['our human S3−S1']).sum() / (B2_GENES['our mouse S3−S1'] ** 2).sum())}
print('Human-on-mouse slope through the origin on the B2 genes (descriptive):', {k: round(v, 3) for k, v in B2_SLOPES.items()})
READING.append('B2: our S3 − S1 is centred per specimen by its median over the universe genes before the species mean '
               '(79 section 3.2); Spearman statistics do not depend on that constant.')
READING.append('B2b partial: rank residuals of x and y on the ranks of Cheval mouse d and our mouse S3 − S1 '
               '(`permutation_spearman` with covariates; residual permutation), as the protocol\'s "same permutation".')

## 6 · B2c · Notebook 79's transfer calls in Cheval

Z-M1, Z-M3, Z-H1 and Z-H3 (no S2 in Cheval). Members ∩ eligible, at least 5. Own-segment sign: S3 programs use d,
S1 programs −d. Per species, the median member contrast m and the share s of positive members, classed by
`library_class` (zonated: m ≥ 0.25, s ≥ 0.7, sign-test p ≤ 0.05; reversed: the mirror; absent: |m| < 0.25;
attenuated otherwise). A call **replicates** when Cheval's source-species class is zonated and its other-species
class equals 79's transfer class.

In [ ]:
def cheval_programs(g):
    eligible = set(g.index[g.eligible])
    rows = []
    for p in B2C_PROGRAMS:
        sign = 1. if SEG[p] == 'S3' else -1.
        members = [x for x in Z79[p] if x in eligible]
        row = {'program': p, 'segment': SEG[p], 'source species': SOURCE[p], 'members': len(Z79[p]),
               'members eligible': len(members), 'genes': ', '.join(members)}
        for sp in ('mouse', 'human'):
            res = pf.library_class(sign * g.loc[members, f'd_{sp}'].to_numpy())
            row.update({f'{sp} {k}': v for k, v in res.items()})
        source, other = row[f'{SOURCE[p]} class'], row[f'{OTHER[p]} class']
        row.update({'Cheval source class': source, 'Cheval other class': other, '79 transfer class': TRANSFER79[p]})
        if 'not testable' in (source, other):
            row['call'] = 'not testable'
        else:
            row['call'] = 'replicates' if source == 'zonated' and other == TRANSFER79[p] else 'does not replicate'
        rows.append(row)
    return pd.DataFrame(rows).set_index('program')


B2C = cheval_programs(CHEVAL)
display(B2C.drop(columns='genes').round(4))
READING.append('B2c: the class uses `library_class` (sign test over members with a non-zero contrast); members are 79\'s '
               'frozen members that are Cheval-eligible.')

## 7 · The atlases (protocol B3)

**Census mouse** (dataset 25818bf7): male donors at "3-6 month-old mature stage" and "12-month-old stage", with at
least 50 cells in each of S1, S2 and S3 (notebook 74's six-donor set).

**Lake human** (KPMP v1.5, normal cortex): the donors with at least 50 nuclei in each of S1, S2 and S3, which are
notebook 38's primary donors. The protocol's file `zonation_check/lake_cortex_s1_s3_all_disease.npz` holds S1 and S3
only (groups `donor|PT-S1`, `donor|PT-S3`, all disease groups). The S1/S2/S3 version of the same Lake pseudobulk is
notebook 75's normal-cortex stage cache (`lake_cortex_pt`, groups `donor|sex|segment`, human genes). It is mapped to
our mouse symbols with notebook 77's rule (Ensembl id first, symbol second, duplicates summed). **Guard:** on every
shared gene and group it equals the npz exactly (sums, totals, nuclei). The npz-only reading (S1 and S3) is a
sensitivity.

**Census human** (dataset 09b518f9): convoluted PT (S1/S2) against S3.

**Per donor:** log2(CPM + 1) per segment (`donor_segment_rates`); the segment contrast (each segment minus the mean
of the others; Census human S3 − PCT), centred per donor by its median over the analysed genes (79 section 3.4).

In [ ]:
import anndata as ad


def newest_cache(folder, stage):
    """A notebook's newest cached payload of one stage (read only), with its key record."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        data = {k: payload[k] for k in payload.files}
    return data, json.loads(Path(str(files[-1]) + '.key.json').read_text())


def census_rates(table, meta, keep, segment_map, segments, min_cells):
    """Per-donor log2(CPM + 1) per segment from Census pseudobulk rows ``keep``."""
    block = meta[keep & meta.cell_type.isin(list(segment_map))]
    frame = pd.DataFrame({'donor': block.donor_id.astype(str).to_numpy(), 'segment': block.cell_type.map(segment_map).to_numpy(),
                          'cells': block.n_cells.to_numpy(float)})
    genes = [g for g in GENES if g in table.columns]
    rates = pf.donor_segment_rates(table.loc[block.index, genes].to_numpy(float), block.total_counts_all_genes.to_numpy(float),
                                   frame, genes, min_cells=min_cells, segments=segments)
    sex = block.groupby('donor_id').sex.first().astype(str)
    cells = frame.pivot_table(index='donor', columns='segment', values='cells', aggfunc='sum', fill_value=0)
    return rates, sex.reindex(list(rates)), cells.reindex(list(rates)), genes


t0 = time.perf_counter()
mouse_table = pd.read_csv(CENSUS / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
mouse_meta = pd.read_csv(CENSUS / 'mouse_pt_segment_pseudobulk_meta.csv', index_col=0)
human_table = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
human_meta = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_meta.csv', index_col=0)
assert mouse_table.index.equals(mouse_meta.index) and human_table.index.equals(human_meta.index)
for table in (mouse_table, human_table):                       # the atlases are on our mouse symbols
    assert len(GENE_SET & set(table.columns)) / len(GENES) > .99

keep_mouse = mouse_meta.sex.eq('male') & mouse_meta.development_stage.isin(MOUSE_STAGES)
CM_RATES, CM_SEX, CM_CELLS, CM_GENES = census_rates(mouse_table, mouse_meta, keep_mouse, CENSUS_MOUSE_SEGMENTS, SEGMENTS, MIN_CELLS)
assert len(CM_RATES) == 6 and (CM_SEX == 'male').all(), CM_SEX
GUARDS.append(f'G6: Census mouse: {len(CM_RATES)} male donors at 3–6 or 12 months, each with ≥ {MIN_CELLS} cells in S1, S2 '
              f'and S3 (cells per segment {int(CM_CELLS.min().min())}–{int(CM_CELLS.max().max())}); {len(CM_GENES):,} of '
              f'{len(GENES):,} universe genes measured')

everyone = pd.Series(True, index=human_meta.index)
CH_RATES, CH_SEX, CH_CELLS, CH_GENES = census_rates(human_table, human_meta, everyone, CENSUS_HUMAN_SEGMENTS, ('PCT', 'S3'), MIN_CELLS)
males = human_meta.sex.eq('male')
CHM_RATES, CHM_SEX, CHM_CELLS, _ = census_rates(human_table, human_meta, males, CENSUS_HUMAN_SEGMENTS, ('PCT', 'S3'), 1)
assert len(CH_RATES) == 6 and len(CHM_RATES) == 6 and (CHM_SEX == 'male').all()
n_male_50 = int((CH_SEX == 'male').sum())
display(pd.concat({'≥ 50 cells (primary)': CH_CELLS.assign(sex=CH_SEX), 'male, no floor (sensitivity)': CHM_CELLS.assign(sex=CHM_SEX)}))
GUARDS.append(f'G6: Census human: 6 donors with ≥ {MIN_CELLS} cells in both labels ({n_male_50} male); the 6 male donors '
              f'without a floor have {int(CHM_CELLS.S3.min())}–{int(CHM_CELLS.S3.max())} S3 cells')
others = CHM_CELLS.drop(index=[d for d in CHM_CELLS.index if d in CH_RATES])
READING.append(f'Census human: the protocol names "6 male donors". The file has 6 male donors, but only {n_male_50} of them '
               f'have ≥ {MIN_CELLS} cells in both labels (the other {len(others)} have {int(others.S3.min())}–{int(others.S3.max())} '
               'S3 cells). The '
               "record's \"Census human 6\" (pt-gene-model-nb.md, pt-package-pseudospace.md) is the 6 donors with ≥ 50 cells "
               'in both labels (2 male, 4 female), which 77\'s `census_donors` rule gives. That set is primary; the 6 male '
               'donors without a cell floor are a sensitivity. Male donors with ≥ 50 cells (2) are below donor_class\'s 3.')

lake, lake_key = newest_cache(T75, 'lake_cortex_pt')
params = str(lake_key['params'])
assert "'disease': 'normal'" in params and 'cortex of kidney' in params and "'labels': 'PT-S1,PT-S2,PT-S3'" in params, params
native = ad.read_h5ad(NATIVE_COR1, backed='r')
human_ids = native.var['gene_ids'].astype(str)
native.file.close()
by_id = dict(zip(orthologs.human_symbol.map(human_ids[~human_ids.index.duplicated()]), orthologs.mouse_symbol))
by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake['var_ids'].astype(str), lake['var_names'].astype(str))])
mapped = target != ''
LAKE_SUMS = pd.DataFrame(lake['sums'][:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
groups75 = pd.Series(lake['groups'].astype(str)).str.split('|', expand=True)
groups75.columns = ['donor', 'sex', 'segment']
key75 = (groups75.donor + '|PT-' + groups75.segment).tolist()

npz = np.load(LAKE_NPZ, allow_pickle=False)
NPZ_GROUPS = npz['groups'].astype(str)
NPZ_SUMS = pd.DataFrame(npz['sums'], columns=npz['target'].astype(str), index=NPZ_GROUPS)
disease = pd.read_csv(LAKE_DONORS).set_index('donor').disease.astype(str)
npz_donor = pd.Series(NPZ_GROUPS).str.split('|').str[0]
npz_normal = [g for g, d in zip(NPZ_GROUPS, npz_donor) if disease.get(d) == 'normal']
assert set(npz_normal) <= set(key75), set(npz_normal) - set(key75)
shared_genes = [x for x in NPZ_SUMS.columns if x in LAKE_SUMS.columns]
L75 = LAKE_SUMS.set_axis(key75, axis=0)
np.testing.assert_array_equal(L75.loc[npz_normal, shared_genes].to_numpy(), NPZ_SUMS.loc[npz_normal, shared_genes].to_numpy())
np.testing.assert_array_equal(pd.Series(lake['totals'], index=key75)[npz_normal].to_numpy(),
                              pd.Series(npz['totals'], index=NPZ_GROUPS)[npz_normal].to_numpy())
np.testing.assert_array_equal(pd.Series(lake['nuclei'], index=key75)[npz_normal].to_numpy(),
                              pd.Series(npz['counts'], index=NPZ_GROUPS)[npz_normal].to_numpy())
lake_genes = [g for g in GENES if g in LAKE_SUMS.columns]
lake_meta = pd.DataFrame({'donor': groups75.donor, 'segment': groups75.segment, 'cells': lake['nuclei'].astype(float)})
LAKE_RATES = pf.donor_segment_rates(LAKE_SUMS[lake_genes].to_numpy(float), lake['totals'], lake_meta, lake_genes,
                                    min_cells=MIN_CELLS, segments=SEGMENTS)
assert sorted(LAKE_RATES) == sorted(LAKE_RECORD), sorted(LAKE_RATES)
assert all(disease.get(d) == 'normal' for d in LAKE_RATES)
LAKE_SEX = groups75.drop_duplicates('donor').set_index('donor').sex.reindex(list(LAKE_RATES))
GUARDS.append(f"G7: Lake: notebook 75's S1/S2/S3 normal-cortex cache, mapped to our symbols by 77's rule, equals the "
              f"protocol's npz on all {len(npz_normal)} normal-donor S1/S3 groups and {len(shared_genes):,} shared genes "
              f'(sums, totals and nuclei exactly); the {len(LAKE_RATES)} donors with ≥ {MIN_CELLS} nuclei in S1, S2 and S3 '
              f"are notebook 38's primary donors ({int((LAKE_SEX == 'male').sum())} male), all 'normal' in lake_donor_amplitude.csv")

# Sensitivity: the npz as given (S1 and S3 only); normal donors with >= 50 nuclei in S1 and S3.
npz_meta = pd.DataFrame({'donor': npz_donor, 'segment': pd.Series(NPZ_GROUPS).str.split('|').str[1].str.replace('PT-', '', regex=False),
                         'cells': npz['counts'].astype(float)})
normal_rows = npz_meta.donor.map(disease).eq('normal').to_numpy()
npz_genes = [g for g in GENES if g in NPZ_SUMS.columns]
LAKE13_RATES = pf.donor_segment_rates(NPZ_SUMS.loc[normal_rows, npz_genes].to_numpy(float), npz['totals'][normal_rows],
                                      npz_meta[normal_rows], npz_genes, min_cells=MIN_CELLS, segments=('S1', 'S3'))
assert set(LAKE_RECORD) <= set(LAKE13_RATES)
print(f'Lake, npz reading (S1 and S3 only): {len(LAKE13_RATES)} normal donors with ≥ {MIN_CELLS} nuclei in S1 and S3')
READING.append("Lake: the protocol's npz holds only S1 and S3 groups (donor|PT-segment, all diseases, no sex). The protocol's "
               "donors (≥ 50 nuclei in each of S1, S2 and S3; notebook 38's primary donors) need S2, so the primary arm uses "
               "notebook 75's S1/S2/S3 normal-cortex cache of the same Lake pseudobulk (guard G7: identical to the npz on "
               "every shared S1/S3 group and gene). 'S2 optional' is read as: S2 programs are tested where the atlas has S2. "
               f'The npz-only reading ({len(LAKE13_RATES)} normal donors with ≥ 50 nuclei in S1 and S3; S3 − S1 contrasts; '
               'S2 programs not testable) is a sensitivity.')
print('\n'.join('· ' + g for g in GUARDS[-3:]))
print(f'Atlases {time.perf_counter() - t0:.0f} s')

In [ ]:
def centred_contrasts(rates_by_donor, kind):
    """{donor: genes x contrasts}, each column centred by its median over the analysed genes (79 section 3.4)."""
    out = {}
    for donor, r in rates_by_donor.items():
        if kind == 'S1/S2/S3':
            c = pf.segment_contrasts(r, SEGMENTS)
        elif kind == 'PCT/S3':
            c = pd.DataFrame({'S3 − PCT': r['S3'] - r['PCT']})
        elif kind == 'S1/S3':
            c = pd.DataFrame({'S3 − S1': r['S3'] - r['S1']})
        else:
            raise ValueError(kind)
        out[donor] = c - c.median()
    return out


def own_segment(contrast, seg, kind):
    """The contrast of a program's own segment: S1/S2/S3 atlases directly; two-label atlases signed by the segment."""
    if kind == 'S1/S2/S3':
        return contrast[seg]
    column = contrast.columns[0]
    if kind == 'S1/S3' and seg == 'S2':
        return None                                     # no S2 in the npz
    return contrast[column] * (1. if seg == 'S3' else -1.)


ATLASES = {
    'Census mouse': {'species': 'mouse', 'kind': 'S1/S2/S3', 'arm': 'primary', 'rates': CM_RATES, 'sex': CM_SEX},
    'Lake human': {'species': 'human', 'kind': 'S1/S2/S3', 'arm': 'primary', 'rates': LAKE_RATES, 'sex': LAKE_SEX},
    'Census human': {'species': 'human', 'kind': 'PCT/S3', 'arm': 'primary', 'rates': CH_RATES, 'sex': CH_SEX},
    'Census human, 6 male donors (no cell floor)': {'species': 'human', 'kind': 'PCT/S3', 'arm': 'sensitivity',
                                                    'rates': CHM_RATES, 'sex': CHM_SEX},
    'Lake human, npz S1/S3 (12 normal donors)': {'species': 'human', 'kind': 'S1/S3', 'arm': 'sensitivity',
                                                  'rates': LAKE13_RATES, 'sex': pd.Series(dtype=str)}}
for name, A in ATLASES.items():
    A['contrasts'] = centred_contrasts(A['rates'], A['kind'])
    A['genes'] = set(next(iter(A['rates'].values())).index)
    A['n'] = len(A['rates'])
display(pd.DataFrame({n: {'species': A['species'], 'kind': A['kind'], 'arm': A['arm'], 'donors': A['n'],
                          'genes': len(A['genes'])} for n, A in ATLASES.items()}).T)

## 8 · B3 · Notebook 79's zonation calls in the atlases, donors as replicates

Per donor and program: the median member contrast of the program's segment against the other segments (Census
human: S3 − PCT, sign-flipped for S1/S2 programs). Class per atlas (`donor_class`): zonated when the donor median
≥ 0.25, ≥ 80% of donors ≥ 0.25 and Wilcoxon signed-rank p ≤ 0.05; reversed the mirror; absent when |donor median| <
0.25; attenuated otherwise. The source-species atlas is the positive control; the other species' atlas tests 79's
transfer class.

In [ ]:
DONOR_ROWS, B3_ROWS = [], []
for atlas, A in ATLASES.items():
    for p in Z_PROGRAMS:
        members = [g for g in Z79[p] if g in A['genes']]
        values = {}
        for donor, c in A['contrasts'].items():
            own = own_segment(c, SEG[p], A['kind'])
            values[donor] = np.nan if own is None else float(np.median(own[members]))
            DONOR_ROWS.append({'atlas': atlas, 'arm': A['arm'], 'species': A['species'], 'donor': donor,
                               'sex': A['sex'].get(donor, ''), 'program': p, 'readout': 'B3 own-segment contrast',
                               'segment': SEG[p], 'members': len(members), 'value': values[donor]})
        res = pf.donor_class(list(values.values()))
        role = 'positive control' if A['species'] == SOURCE[p] else 'transfer'
        ours = SOURCE79[p] if role == 'positive control' else TRANSFER79[p]
        agrees = np.nan if res['class'] == 'not testable' else bool(res['class'] == ours)
        B3_ROWS.append({'readout': 'B3', 'atlas': atlas, 'arm': A['arm'], 'program': p, 'segment': SEG[p],
                        'source species': SOURCE[p], 'role': role, 'members': len(members), **res,
                        '79 class': ours, 'agrees with 79': agrees})
B3 = pd.DataFrame(B3_ROWS)
display(B3[B3.arm.eq('primary')].round(4))
transfer_primary = B3[B3.arm.eq('primary') & B3.role.eq('transfer')]
B3_AGREEMENT = transfer_primary.groupby('atlas')['agrees with 79'].agg(lambda s: f'{int(s.sum())} of {int(s.notna().sum())}')
print('Agreement of 79\'s transfer class with the atlas class (primary arms):')
display(B3_AGREEMENT)
display(B3[B3.arm.eq('sensitivity')].round(4))
READING.append('B3 contrasts are centred per donor by their median over the universe genes measured in that atlas. Members '
               'are 79\'s frozen members measured in the atlas; no detection filter is applied (a member absent from a '
               'donor contributes a contrast of 0 before centring).')
READING.append('B3 "agreement" is equality of classes; attenuated and absent agree only with themselves (B2c\'s rule). '
               'Positive-control rows compare with 79\'s source class (zonated for all six).')

## 9 · A3 (atlas half) · Do CellSpectra's conserved PT functions peak in the same segment?

For each function, per specimen (ours) or donor (atlas): the program median of the segment contrast in S1, S2 and
S3, and its peak segment.
- **Ours:** the peak of the mean of the species' two specimens; the per-specimen peaks are shown.
- **Atlas:** the peak of the per-segment median over donors; the share of donors that peak there.
- **Readout:** atlas peak = our peak, per species (Census mouse against our mice, Lake against our human sections).

In [ ]:
A3_ROWS = []
for f in FUNCTIONS:
    for species, atlas in (('mouse', 'Census mouse'), ('human', 'Lake human')):
        ours = np.array([our_program(FN[f], species, seg) for seg in SEGMENTS])        # segments x specimens
        ours_mean = ours.mean(axis=1)
        spec_peaks = [SEGMENTS[int(np.argmax(ours[:, j]))] for j in range(ours.shape[1])]
        A = ATLASES[atlas]
        members = [g for g in FN[f] if g in A['genes']]
        donor_values = {}
        for donor, c in A['contrasts'].items():
            donor_values[donor] = [float(np.median(c.loc[members, seg])) for seg in SEGMENTS]
            for seg, v in zip(SEGMENTS, donor_values[donor]):
                DONOR_ROWS.append({'atlas': atlas, 'arm': A['arm'], 'species': species, 'donor': donor,
                                   'sex': A['sex'].get(donor, ''), 'program': f, 'readout': 'A3 segment contrast',
                                   'segment': seg, 'members': len(members), 'value': v})
        frame = pd.DataFrame(donor_values, index=list(SEGMENTS))
        atlas_median = frame.median(axis=1)
        atlas_peak = SEGMENTS[int(np.argmax(atlas_median.to_numpy()))]
        donor_peaks = frame.idxmax(axis=0)
        A3_ROWS.append({'readout': 'A3 placement', 'program': f, 'species': species, 'atlas': atlas,
                        'members (ours)': len(FN[f]), 'members (atlas)': len(members),
                        **{f'ours {s}': ours_mean[k] for k, s in enumerate(SEGMENTS)},
                        'ours peak': SEGMENTS[int(np.argmax(ours_mean))], 'ours range': float(np.ptp(ours_mean)),
                        'ours peaks per specimen': ', '.join(spec_peaks),
                        'specimens agree': len(set(spec_peaks)) == 1,
                        **{f'atlas {s}': atlas_median[s] for s in SEGMENTS}, 'atlas peak': atlas_peak,
                        'atlas range': float(np.ptp(atlas_median.to_numpy())),
                        'donors at the atlas peak': f'{int((donor_peaks == atlas_peak).sum())} of {len(donor_peaks)}',
                        'atlas peak = ours': atlas_peak == SEGMENTS[int(np.argmax(ours_mean))]})
A3 = pd.DataFrame(A3_ROWS)
peaks = A3.pivot_table(index='program', columns='species', values='atlas peak', aggfunc='first').reindex(FUNCTIONS)
A3['atlas peaks equal between species'] = A3.program.map(peaks.mouse.eq(peaks.human))
ours_peaks = A3.pivot_table(index='program', columns='species', values='ours peak', aggfunc='first').reindex(FUNCTIONS)
A3['our peaks equal between species'] = A3.program.map(ours_peaks.mouse.eq(ours_peaks.human))
display(A3.round(3))
print(f"A3: atlas peak = our peak in {int(A3['atlas peak = ours'].sum())} of {len(A3)} function × species cells")
READING.append('A3: the range (max − min over S1–S3 of the program medians) is shown beside each peak as a descriptive '
               'size; the protocol fixes no amplitude threshold for a peak, so none is applied.')
READING.append('A3 (atlas half): our per-species peak is the argmax of the mean of the two specimens\' program medians of '
               'the segment contrast; the atlas peak is the argmax of the per-segment median over donors. Census human '
               '(two labels) is not used for A3, as the protocol names Census mouse and Lake.')

## 10 · Sensitivity: one Cheval tag = the column's most frequent non-zero value

The protocol's rule takes the smallest non-zero value as one tag. In each used column a few values lie below the
modal value (section 3), so the smallest value is not one tag. Every Cheval readout is rerun with the modal unit.

In [ ]:
B1_BINS_M, B1_SLOPES_M, B1_VERDICT_M = b1(CHEVAL_MODAL)
B2_M, _ = versus_ours(CHEVAL_MODAL)
B2C_M = cheval_programs(CHEVAL_MODAL)
print({'eligible (modal unit)': int(CHEVAL_MODAL.eligible.sum()), 'eligible (protocol)': int(CHEVAL.eligible.sum())})
display(B1_BINS_M.round(4))
display(B1_SLOPES_M.round(3))
display(B2_M.round(4))
display(B2C_M[['members eligible', 'mouse class', 'mouse median', 'human class', 'human median', 'call']].round(3))
SENS_CHANGES = []
for sel in ('mouse', 'human'):
    for key in B1_VERDICT[sel]:
        if B1_VERDICT[sel][key] != B1_VERDICT_M[sel][key]:
            SENS_CHANGES.append(f'B1a {sel}-selected "{key}": {B1_VERDICT[sel][key]} → {B1_VERDICT_M[sel][key]}')
for key in B1_VERDICT['B1b']:
    if B1_VERDICT['B1b'][key] != B1_VERDICT_M['B1b'][key]:
        SENS_CHANGES.append(f'B1b "{key}": {B1_VERDICT["B1b"][key]} → {B1_VERDICT_M["B1b"][key]}')
for i in (2, 3):
    if B2.loc[i, 'claim'] != B2_M.loc[i, 'claim']:
        SENS_CHANGES.append(f"{B2.loc[i, 'readout']}: {B2.loc[i, 'claim']} → {B2_M.loc[i, 'claim']}")
for p in B2C_PROGRAMS:
    if B2C.loc[p, 'call'] != B2C_M.loc[p, 'call']:
        SENS_CHANGES.append(f"B2c {p}: {B2C.loc[p, 'call']} → {B2C_M.loc[p, 'call']}")
print('Calls changed by the modal unit:', SENS_CHANGES or 'none')

## 10b · Post hoc (outside the frozen protocol): the single-gene hypotheses in Cheval and the atlases

Added after notebook 81, to ask whether its single-gene species differences are species biology or our specimens.
For each gene of `genes_lit.csv` (C2, D, F; eligible), the S3 − S1 contrast (log2) in every data set: ours per
specimen (centred as in B2), Cheval's d (SAGE tags), and each atlas donor's S3 − S1 (Census human S3 − PCT),
centred per donor by its median over the analysed genes. Descriptive only: no test, no call.

In [ ]:
GENES_LIT = pd.read_csv(LIT / 'genes_lit.csv')
assert hashlib.sha256((LIT / 'genes_lit.csv').read_bytes()).hexdigest() == \
    '111cca20e92708f7fbceccca6b5fa0d49453544bfb5a57c26336e8e3805a22d8'
SINGLE = [g for g in GENES_LIT.gene[GENES_LIT.status.eq('eligible')] if g in GENE_SET]


def donor_s3_minus(rates_by_donor, base):
    """Donors x genes: S3 − ``base`` per donor, centred by the donor's median over the analysed genes."""
    out = {}
    for donor, r in rates_by_donor.items():
        c = r['S3'] - r[base]
        out[donor] = c - c.median()
    return pd.DataFrame(out)


POSTHOC = S3_S1.loc[SINGLE].add_prefix('ours ')
cheval_tags = CHEVAL.reindex(SINGLE)
POSTHOC['Cheval mouse d'] = cheval_tags.d_mouse
POSTHOC['Cheval human d'] = cheval_tags.d_human
POSTHOC['Cheval tags (mouse S1, S3; human S1, S3)'] = [
    '' if np.isnan(r.c_mouse_S1) else f'{r.c_mouse_S1:.0f}, {r.c_mouse_S3:.0f}; {r.c_human_S1:.0f}, {r.c_human_S3:.0f}'
    for r in cheval_tags.itertuples()]
for name in ('Census mouse', 'Lake human', 'Census human'):
    A = ATLASES[name]
    d = donor_s3_minus(A['rates'], 'PCT' if A['kind'] == 'PCT/S3' else 'S1').reindex(SINGLE)
    POSTHOC[f'{name} median'] = d.median(axis=1)
    POSTHOC[f'{name} donors with the median sign'] = [f'{int((np.sign(r) == np.sign(r.median())).sum())}/{r.notna().sum()}'
                                                      for _, r in d.iterrows()]
display(POSTHOC.round(2))
READING.append('Section 10b is post hoc (added after notebook 81 and outside the frozen protocol): descriptive S3 − S1 '
               'contrasts of the single-gene hypotheses in every data set; it changes no pre-specified readout.')

## 11 · Figures

- **fig82a:** B1a, the share of zonated genes whose other-species contrast has the same sign, by strength bin, for
  both selection directions (95% Clopper–Pearson intervals).
- **fig82b:** B2, Cheval against ours: the two positive controls, the interaction, and the interaction's rank
  residuals given both mouse contrasts (beyond flattening).
- **fig82c:** the zonation programs' own-segment contrast per data set (cell colour) with its class, and the A3
  functions' segment contrasts in our data and the atlases.

In [ ]:
from matplotlib.colors import TwoSlopeNorm

fig, ax = plt.subplots(figsize=(70 * MM, 52 * MM))
x = np.arange(len(STRENGTH_BINS) - 1)
for k, (sel, color) in enumerate((('mouse', MOUSE_COLOR), ('human', HUMAN_COLOR))):
    block = B1_BINS[B1_BINS.selection.eq(f'zonated in {sel}')]
    ok = block.n.to_numpy() > 0
    xs = x[ok] + (k - .5) * .18
    share = block.share.to_numpy()[ok]
    ax.errorbar(xs, share, yerr=[share - block['ci low'].to_numpy()[ok], block['ci high'].to_numpy()[ok] - share],
                fmt='o-', color=color, ms=4, lw=1, capsize=0, label=f'selected in {sel}')
ax.axhline(.5, color=MUTED, lw=.6, ls='--', zorder=0)
ax.set_xticks(x)
ax.set_xticklabels([f'[{lo:g}, {hi:g})' for lo, hi in zip(STRENGTH_BINS[:-1], STRENGTH_BINS[1:])])
ax.set_xlabel('|S3 − S1| in the selecting species (log2)')
ax.set_ylabel('same sign in the other species')
ax.set_ylim(0, 1.02)
ax.legend(loc='lower right')
save(fig, 'fig82a_cheval_strength')

In [ ]:
from scipy.stats import rankdata


def rank_residual(v, cov):
    r = rankdata(v)
    z = np.column_stack([np.ones(len(r))] + [rankdata(c) for c in cov.T])
    return r - z @ np.linalg.lstsq(z, r, rcond=None)[0]


G = B2_GENES
fig, axes = plt.subplots(1, 4, figsize=(183 * MM, 46 * MM))
panels = [('cheval d_mouse', 'our mouse S3−S1', MOUSE_COLOR, 'Cheval mouse S3 − S1 (log2)', 'our mouse S3 − S1 (log2)'),
          ('cheval d_human', 'our human S3−S1', HUMAN_COLOR, 'Cheval human S3 − S1 (log2)', 'our human S3 − S1 (log2)'),
          ('cheval interaction', 'our interaction', INK, 'Cheval (human − mouse) S3 − S1', 'our (human − mouse) S3 − S1')]
for ax, (cx, cy, color, xl, yl) in zip(axes[:3], panels):
    ax.scatter(G[cx], G[cy], s=5, color=color, alpha=.7, lw=0)
    ax.axhline(0, color=LIGHT, lw=.6, zorder=0)
    ax.axvline(0, color=LIGHT, lw=.6, zorder=0)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
cov = np.column_stack([G['cheval d_mouse'], G['our mouse S3−S1']])
axes[3].scatter(rank_residual(G['cheval interaction'].to_numpy(), cov), rank_residual(G['our interaction'].to_numpy(), cov),
                s=5, color=INK, alpha=.7, lw=0)
axes[3].axhline(0, color=LIGHT, lw=.6, zorder=0)
axes[3].axvline(0, color=LIGHT, lw=.6, zorder=0)
axes[3].set_xlabel('Cheval interaction, rank residual')
axes[3].set_ylabel('our interaction, rank residual')
for ax, title in zip(axes, ('B2a mouse', 'B2a human', 'B2b interaction', 'B2b beyond flattening')):
    ax.set_title(title, loc='left')
fig.tight_layout()
save(fig, 'fig82b_cheval_vs_ours')

In [ ]:
CODES = {'zonated': 'Z', 'attenuated': 'at', 'absent': 'ab', 'reversed': 'R', 'not testable': 'nt'}
columns = [('ours', 'mouse'), ('ours', 'human'), ('Cheval', 'mouse'), ('Cheval', 'human'),
           ('Census mouse', 'mouse'), ('Lake human', 'human'), ('Census human', 'human')]
value = np.full((len(Z_PROGRAMS), len(columns)), np.nan)
code = np.full(value.shape, '', dtype=object)
for i, p in enumerate(Z_PROGRAMS):
    for j, (source, species) in enumerate(columns):
        if source == 'ours':
            value[i, j] = float(np.mean(our_program(Z79[p], species, SEG[p])))
            code[i, j] = CODES[SOURCE79[p] if species == SOURCE[p] else TRANSFER79[p]]
        elif source == 'Cheval':
            if p in B2C.index:
                value[i, j] = B2C.loc[p, f'{species} median']
                code[i, j] = CODES[B2C.loc[p, f'{species} class']]
        else:
            r = B3[B3.atlas.eq(source) & B3.program.eq(p)].iloc[0]
            value[i, j] = r['median']
            code[i, j] = CODES[r['class']]
fig, axes = plt.subplots(1, 2, figsize=(183 * MM, 62 * MM), gridspec_kw={'width_ratios': [1, 1.55]})
ax = axes[0]
norm = TwoSlopeNorm(0, -1.5, 1.5)
im = ax.imshow(np.clip(value, -1.5, 1.5), cmap='RdBu_r', norm=norm, aspect='auto')
for i in range(value.shape[0]):
    for j in range(value.shape[1]):
        if not np.isfinite(value[i, j]):
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color=LIGHT, lw=0))
        if code[i, j]:
            ink = 'white' if np.isfinite(value[i, j]) and abs(value[i, j]) > 1 else INK
            ax.text(j, i, code[i, j], ha='center', va='center', fontsize=5.5, color=ink)
        if columns[j][1] == SOURCE[Z_PROGRAMS[i]] and np.isfinite(value[i, j]):
            ax.add_patch(plt.Rectangle((j - .48, i - .48), .96, .96, fill=False, ec=INK, lw=.8))
ax.set_xticks(range(len(columns)))
ax.set_xticklabels([f'{s}\n({sp})' if s in ('ours', 'Cheval') else s.replace(' ', '\n') for s, sp in columns])
ax.set_yticks(range(len(Z_PROGRAMS)))
ax.set_yticklabels([f'{p} ({SEG[p]}, {SOURCE[p]})' for p in Z_PROGRAMS])
fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.05, pad=.2, aspect=25, label='own-segment contrast (log2)')

ax = axes[1]
blocks = [('ours', 'mouse'), ('Census mouse', 'mouse'), ('ours', 'human'), ('Lake human', 'human')]
grid = np.full((len(FUNCTIONS), 3 * len(blocks)), np.nan)
peaks_marks = []
for i, f in enumerate(FUNCTIONS):
    for b, (source, species) in enumerate(blocks):
        r = A3[A3.program.eq(f) & A3.species.eq(species)].iloc[0]
        prefix = 'ours' if source == 'ours' else 'atlas'
        vals = [r[f'{prefix} {s}'] for s in SEGMENTS]
        grid[i, 3 * b:3 * b + 3] = vals
        peaks_marks.append((3 * b + int(np.argmax(vals)), i))
im2 = ax.imshow(np.clip(grid, -1, 1), cmap='RdBu_r', norm=TwoSlopeNorm(0, -1, 1), aspect='auto')
for xm, ym in peaks_marks:
    ax.plot(xm, ym, 'o', ms=2.5, color=INK)
for b in range(1, len(blocks)):
    ax.axvline(3 * b - .5, color='white', lw=2)
ax.set_xticks(range(grid.shape[1]))
ax.set_xticklabels(list(SEGMENTS) * len(blocks))
top = ax.secondary_xaxis('top')
top.set_xticks([3 * b + 1 for b in range(len(blocks))])
top.set_xticklabels([source if source != 'ours' else f'ours ({species})' for source, species in blocks])
top.tick_params(length=0)
top.spines['top'].set_visible(False)
ax.set_yticks(range(len(FUNCTIONS)))
ax.set_yticklabels(FUNCTIONS)
fig.colorbar(im2, ax=ax, orientation='horizontal', fraction=.05, pad=.2, aspect=35,
             label='segment contrast, program median (log2); dot = peak segment')
fig.tight_layout()
save(fig, 'fig82c_program_classes')
READING.append('fig82c: cell letters are classes (Z zonated, at attenuated, ab absent, R reversed, nt not testable); '
               'boxed cells are the source species; grey cells have no S2 in Cheval; "ours" values are the mean of the two specimens\' program medians with '
               "79's classes; atlas values are the donor median.")

## 12 · Outputs

In [ ]:
cheval_out = CHEVAL.copy()
cheval_out['in universe'] = cheval_out.index.isin(GENE_SET)
cheval_out['label/orientation gene'] = cheval_out.index.isin(LABEL_ORIENTATION)
cheval_out['in B2'] = cheval_out.index.isin(B2_GENES.index)
cheval_out = cheval_out.join(OURS.rename(columns={'mouse': 'our mouse S3−S1', 'human': 'our human S3−S1'}), how='left')
cheval_out = cheval_out.join(CHEVAL_MODAL[['d_human', 'd_mouse', 'eligible']].add_suffix(' (modal unit)'))
cheval_out.to_csv(tables / 'cheval_genes.csv')
strength = pd.concat({'protocol unit': pd.concat([B1_BINS.assign(readout='B1a'),
                                                  B1_SLOPES.reset_index().assign(readout='B1b')], ignore_index=True),
                      'modal unit (sensitivity)': pd.concat([B1_BINS_M.assign(readout='B1a'),
                                                             B1_SLOPES_M.reset_index().assign(readout='B1b')], ignore_index=True)},
                     names=['arm', 'row'])
strength.to_csv(tables / 'cheval_strength.csv')
pd.concat({'protocol unit': B2, 'modal unit (sensitivity)': B2_M}, names=['arm', 'row']).to_csv(tables / 'cheval_vs_ours.csv')
pd.concat({'protocol unit': B2C, 'modal unit (sensitivity)': B2C_M}, names=['arm', 'program']).to_csv(tables / 'cheval_programs.csv')
DONOR_PROGRAMS = pd.DataFrame(DONOR_ROWS)
DONOR_PROGRAMS.to_csv(tables / 'atlas_donor_programs.csv', index=False)
ATLAS_CLASSES = pd.concat([B3, A3.assign(arm='primary')], ignore_index=True)
ATLAS_CLASSES.to_csv(tables / 'atlas_classes.csv', index=False)
POSTHOC.to_csv(tables / 'posthoc_single_genes.csv')
print('Tables:', sorted(p.name for p in tables.iterdir()))

## 13 · Readout of every pre-specified question

In [ ]:
def bins_text(frame):
    return '; '.join(f"{r['bin']} {int(r['agree'])}/{int(r['n'])} ({r['share']:.2f}, p {r['p (> 0.5)']:.3g})" if r['n'] else f"{r['bin']} 0"
                     for _, r in frame.iterrows())


def yes(flags):
    return ', '.join(f'{k}: {"yes" if v else "no"}' for k, v in flags.items())


READOUT = []
for sel in ('mouse', 'human'):
    block = B1_BINS[B1_BINS.selection.eq(f'zonated in {sel}')]
    READOUT.append({'readout': f'B1a, selected in {sel}', 'result': f"{bins_text(block)}; pooled {block['pooled share'].iloc[0]:.2f} "
                    f"(p {block['pooled p (> 0.5)'].iloc[0]:.3g}); trend p {block['trend p (rising)'].iloc[0]:.3g}",
                    'call': yes(B1_VERDICT[sel])})
hm, mh = B1_SLOPES.loc['human d on mouse d'], B1_SLOPES.loc['mouse d on human d']
READOUT.append({'readout': 'B1b slopes', 'result': f"human on mouse {hm['estimate']:.2f} ({hm['ci low']:.2f}–{hm['ci high']:.2f}; "
                f"{int(hm['genes'])} genes); mouse on human {mh['estimate']:.2f} ({mh['ci low']:.2f}–{mh['ci high']:.2f}; "
                f"{int(mh['genes'])} genes); atlas 0.12 and 0.62", 'call': yes(B1_VERDICT['B1b'])})
for _, r in B2.iterrows():
    claim = r.get('claim')
    READOUT.append({'readout': r['readout'], 'result': f"ρ {r['rho']:.3f}, p {r['p']:.3g}, {r['genes']} genes",
                    'call': claim if isinstance(claim, str) else ''})
for p, r in B2C.iterrows():
    READOUT.append({'readout': f'B2c {p}', 'result': f"{r['members eligible']} members; mouse {r['mouse class']} (m {r['mouse median']:.2f}, "
                    f"s {r['mouse share']:.2f}); human {r['human class']} (m {r['human median']:.2f}, s {r['human share']:.2f}); "
                    f"79 transfer {r['79 transfer class']}", 'call': r['call']})
for _, r in B3[B3.arm.eq('primary')].iterrows():
    READOUT.append({'readout': f"B3 {r['program']} in {r['atlas']} ({r['role']})",
                    'result': f"{r['class']} (median {r['median']:.2f}, share {r['share']:.2f}, p {r['p']:.3g}, {r['donors']} donors)",
                    'call': f"79 {r['79 class']}; agrees: {r['agrees with 79']}"})
for _, r in A3.iterrows():
    READOUT.append({'readout': f"A3 {r['program']} ({r['species']}, {r['atlas']})",
                    'result': f"atlas peak {r['atlas peak']} ({r['donors at the atlas peak']} donors; range {r['atlas range']:.2f}); "
                              f"ours {r['ours peak']} (specimens {r['ours peaks per specimen']}; range {r['ours range']:.2f})",
                    'call': f"agrees: {r['atlas peak = ours']}"})
READOUT = pd.DataFrame(READOUT)
display(READOUT)
abandon_b2b = not (B2.loc[2, 'rho'] > 0 and B2.loc[2, 'p'] <= ALPHA)
print('Abandonment rule B2b (the raw ρ fails):', 'met — every positional species claim must say it does not replicate in '
      'microdissected tissue at SAGE depth' if abandon_b2b else 'not met')

## 14 · How the protocol was read

In [ ]:
READING.extend([
    f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, as the protocol names it.",
    'All rules come from `published_findings` (sage_counts, sage_contrast, zonated, strength_agreement, cochran_armitage, '
    'slope_through_origin, permutation_spearman, library_class, donor_class, segment_contrasts, donor_segment_rates, '
    'spearman); the only notebook-local rule is the modal-unit sensitivity.',
    'Notebook 82 fits no model and nothing is expensive, so no stage cache is used; every guard reruns.',
])
HOW = pd.Series(READING, name='how the protocol was read')
HOW.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
for line in READING:
    print('·', line)

## 15 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-published-findings.md (commit 0a44dec)',
          'pass1_fingerprint': FINGERPRINT, 'sha256': {relative(k): v for k, v in SHA.items()},
          'units': int(len(SPECIMEN)), 'genes': len(GENES), 'cheval': cheval_counts,
          'cheval_units': {c: float(1e4 / CHEVAL_LIBRARY[c]) for c in CHEVAL_USED},
          'cheval_units_modal': {c: float(1e4 / CHEVAL_LIBRARY_MODAL[c]) for c in CHEVAL_USED},
          'atlas_donors': {n: {'donors': sorted(A['rates']), 'arm': A['arm']} for n, A in ATLASES.items()},
          'guards': GUARDS, 'readouts': READOUT.to_dict('records'), 'b1_verdict': B1_VERDICT,
          'sensitivity_changes_modal_unit': SENS_CHANGES, 'b2b_abandonment_rule_met': bool(abandon_b2b),
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
print(f"Run record written · {record['seconds']} s")